# 7-day take-up propensity (XGBoost)

Simple baseline using `customer_model_table.csv`. Uses the dataset's **train / val / test** column (no random resplit).

Change `PRODUCT` below to model EV Finance, Green Home, Solar, or Paperless. Target is the matching **`y_*_7d`** label; rows are filtered to **`elig_* = 1`** only (see README).

Install (first code cell): `%pip install -q pandas numpy scikit-learn xgboost shap joblib matplotlib`  
**macOS + XGBoost:** if import fails with `libomp.dylib`, run `brew install libomp` in Terminal, then **Restart kernel** and run again. If Homebrew is not installed, the notebook falls back to sklearn `HistGradientBoostingClassifier` so the rest still runs.

In [6]:
%pip install -q pandas numpy scikit-learn xgboost shap joblib matplotlib

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 7.8 MB 5.4 MB/s eta 0:00:01     |█████████████████████▎          | 5.2 MB 5.4 MB/s eta 0:00:01
     |████████████████████████████████| 249 kB 12.9 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 14.3 MB/s eta 0:00:01     |█████████████████████████▏      | 4.2 MB 14.3 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 7.4 MB/s  eta 0:00:01
     |████████████████████████████████| 126 kB 20.3 MB/s eta 0:00:01
     |████████████████████████████████| 2.9 MB 14.4 MB/s eta 0:00:01
     |████████████████████████████████| 4.7 MB 9.6 MB/s eta 0:00:01     |█████                       

In [8]:
from __future__ import annotations

import ctypes
import shutil
import subprocess
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    precision_recall_curve,
    recall_score,
    roc_auc_score,
    roc_curve,
)

BASE_DIR = Path("/Users/nikhilsingh.parihar/Documents/Megathon")
DATA_PATH = BASE_DIR / "green_customer_dataset/1_modelling_table/customer_model_table.csv"
OUT_DIR = BASE_DIR / "propensity_model_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRODUCT = "ev_finance"  # ev_finance | green_home | solar | paperless

PRODUCT_MAP = {
    "ev_finance": {"label": "y_ev_finance_7d", "elig": "elig_ev_finance"},
    "green_home": {"label": "y_green_home_7d", "elig": "elig_green_home"},
    "solar": {"label": "y_solar_7d", "elig": "elig_solar"},
    "paperless": {"label": "y_paperless_7d", "elig": "elig_paperless"},
}

LABEL_COL = PRODUCT_MAP[PRODUCT]["label"]
ELIG_COL = PRODUCT_MAP[PRODUCT]["elig"]
ARTIFACT_PREFIX = OUT_DIR / PRODUCT


def _load_mac_libomp() -> bool:
    if sys.platform != "darwin":
        return False
    for lib_path in (
        Path("/opt/homebrew/opt/libomp/lib/libomp.dylib"),
        Path("/usr/local/opt/libomp/lib/libomp.dylib"),
    ):
        if lib_path.exists():
            ctypes.CDLL(str(lib_path))
            print(f"Loaded OpenMP library: {lib_path}")
            return True
    return False


if sys.platform == "darwin" and not _load_mac_libomp():
    brew = shutil.which("brew")
    if brew:
        print("Installing libomp via Homebrew (needed for XGBoost on Mac)...")
        subprocess.run([brew, "install", "libomp"], check=False)
        _load_mac_libomp()

USE_XGB = True
xgb = None
try:
    import xgboost as xgb

    print("XGBoost version:", xgb.__version__)
except Exception as exc:
    USE_XGB = False
    from sklearn.ensemble import HistGradientBoostingClassifier

    print("XGBoost could not be loaded:", exc)
    print("Using HistGradientBoostingClassifier instead.")
    print("For XGBoost: brew install libomp  →  Restart kernel  →  re-run notebook.")

XGBoost could not be loaded: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/nikhilsingh.parihar/Library/Python/3.9/lib/python/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <89AD948E-E564-3266-867D-7AF89D6488F0> /Users/nikhilsingh.parihar/Library/Python/3.9/lib/python/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/o

In [9]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
print("Label:", LABEL_COL, "| positive rate (all rows):", df[LABEL_COL].mean())

(20000, 116)
Label: y_ev_finance_7d | positive rate (all rows): 0.0109


In [10]:
label_cols = [c for c in df.columns if c.startswith("y_")]
elig_cols = [c for c in df.columns if c.startswith("elig_")]
drop_cols = {"customer_id", "split"} | set(label_cols) | set(elig_cols)

model_df = df[df[ELIG_COL] == 1].copy()
print(f"Eligible rows: {len(model_df):,} / {len(df):,}")

train_df = model_df[model_df["split"] == "train"].copy()
val_df = model_df[model_df["split"] == "val"].copy()
test_df = model_df[model_df["split"] == "test"].copy()
print(train_df.shape, val_df.shape, test_df.shape)
print("Train positive rate:", train_df[LABEL_COL].mean())

Eligible rows: 19,095 / 20,000
(13397, 116) (2850, 116) (2848, 116)
Train positive rate: 0.01067403149958946


In [12]:
CAT_COLS = [
    "income_band",
    "region",
    "postcode_area",
    "digital_usage",
    "vehicle_type",
    "fuel_type",
]
NUM_COLS = [c for c in train_df.columns if c not in drop_cols and c not in CAT_COLS]


def prepare_features(frame: pd.DataFrame, medians: pd.Series, dummy_columns: list[str]) -> pd.DataFrame:
    x = frame[NUM_COLS + CAT_COLS].copy()
    for col in NUM_COLS:
        x[col] = x[col].astype(float).fillna(medians[col])
    for col in CAT_COLS:
        x[col] = x[col].fillna("__MISSING__").astype(str)
    x = pd.get_dummies(x, columns=CAT_COLS, drop_first=False)
    x = x.reindex(columns=dummy_columns, fill_value=0)
    return x.astype(np.float64)


train_medians = train_df[NUM_COLS].median(numeric_only=True)

train_raw = train_df[NUM_COLS + CAT_COLS].copy()
for col in NUM_COLS:
    train_raw[col] = train_raw[col].astype(float).fillna(train_medians[col])
for col in CAT_COLS:
    train_raw[col] = train_raw[col].fillna("__MISSING__").astype(str)
dummy_columns = pd.get_dummies(train_raw, columns=CAT_COLS, drop_first=False).columns.tolist()

X_train = prepare_features(train_df, train_medians, dummy_columns)
X_val = prepare_features(val_df, train_medians, dummy_columns)
X_test = prepare_features(test_df, train_medians, dummy_columns)

y_train = train_df[LABEL_COL].astype(int).values
y_val = val_df[LABEL_COL].astype(int).values
y_test = test_df[LABEL_COL].astype(int).values

print(X_train.shape, "features")

(13397, 234) features


In [13]:
scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

if USE_XGB:
    model = xgb.XGBClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=50,
    )
else:
    model = HistGradientBoostingClassifier(
        max_iter=800,
        max_depth=5,
        learning_rate=0.05,
        random_state=42,
        class_weight="balanced",
    )
    model.fit(X_train, y_train)

print("Trained:", type(model).__name__)

Trained: HistGradientBoostingClassifier


In [14]:
THRESHOLD = 0.5


def classification_metrics(y_true, y_prob, threshold=0.5) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    out = {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "positives": int(y_true.sum()),
        "n": int(len(y_true)),
    }
    return out


def print_metrics(name: str, y_true, y_prob) -> pd.Series:
    m = classification_metrics(y_true, y_prob, THRESHOLD)
    row = pd.Series(m, name=name)
    print(name)
    for k, v in m.items():
        if k in {"n", "positives"}:
            print(f"  {k}: {v}")
        else:
            print(f"  {k}: {v:.4f}")
    return row


p_train = model.predict_proba(X_train)[:, 1]
p_val = model.predict_proba(X_val)[:, 1]
p_test = model.predict_proba(X_test)[:, 1]

metrics_rows = [
    print_metrics("train", y_train, p_train),
    print_metrics("val", y_val, p_val),
    print_metrics("test", y_test, p_test),
]
metrics_df = pd.DataFrame(metrics_rows)
metrics_df

train
  roc_auc: 0.9417
  pr_auc: 0.3124
  accuracy: 0.7907
  precision: 0.0457
  recall: 0.9371
  f1: 0.0872
  positives: 143
  n: 13397
val
  roc_auc: 0.7617
  pr_auc: 0.0435
  accuracy: 0.7807
  precision: 0.0272
  recall: 0.5000
  f1: 0.0516
  positives: 34
  n: 2850
test
  roc_auc: 0.7321
  pr_auc: 0.0334
  accuracy: 0.7823
  precision: 0.0323
  recall: 0.4878
  f1: 0.0606
  positives: 41
  n: 2848


,roc_auc,pr_auc,accuracy,precision,recall,f1,positives,n
train,0.941687,0.312424,0.790699,0.045749,0.937063,0.087240,143.0,13397.0
val,0.761693,0.043521,0.780702,0.027200,0.500000,0.051593,34.0,2850.0
test,0.732124,0.033404,0.782303,0.032310,0.487805,0.060606,41.0,2848.0


In [ ]:
def assign_deciles_equal_count(scores: np.ndarray) -> np.ndarray:
    """Decile 1 = highest propensity; each decile has ~equal number of rows."""
    n = len(scores)
    order = np.argsort(-scores, kind="mergesort")
    decile = np.empty(n, dtype=int)
    for d in range(10):
        lo = int(d * n / 10)
        hi = int((d + 1) * n / 10)
        decile[order[lo:hi]] = d + 1
    return decile


def decile_mins_from_reference(reference_scores: np.ndarray) -> np.ndarray:
    """Minimum score in each decile (1 = highest group), from equal-count split on reference."""
    ref_dec = assign_deciles_equal_count(reference_scores)
    return np.array([reference_scores[ref_dec == d].min() for d in range(1, 11)])


def scores_to_decile(scores: np.ndarray, decile_mins: np.ndarray) -> np.ndarray:
    """Apply validation decile score floors (decile 1 if score >= mins[0], etc.)."""
    out = np.full(len(scores), 10, dtype=int)
    for i, s in enumerate(scores):
        for d in range(1, 11):
            if s >= decile_mins[d - 1]:
                out[i] = d
                break
    return out


def decile_report(
    y_true: np.ndarray, scores: np.ndarray, decile: np.ndarray, split_name: str
) -> pd.DataFrame:
    base_rate = y_true.mean()
    rows = []
    for d in range(1, 11):
        mask = decile == d
        conv = y_true[mask].mean() if mask.any() else np.nan
        lift = conv / base_rate if base_rate > 0 else np.nan
        rows.append(
            {
                "split": split_name,
                "decile": d,
                "n": int(mask.sum()),
                "conversion_rate": conv,
                "lift": lift,
                "score_min": float(scores[mask].min()) if mask.any() else np.nan,
                "score_max": float(scores[mask].max()) if mask.any() else np.nan,
            }
        )
    return pd.DataFrame(rows)


# Equal-count deciles on validation; save score floors for test scoring
decile_mins = decile_mins_from_reference(p_val)
val_decile = assign_deciles_equal_count(p_val)

val_deciles = decile_report(y_val, p_val, val_decile, "val")
test_decile = scores_to_decile(p_test, decile_mins)
test_deciles = decile_report(y_test, p_test, test_decile, "test")

overall_val_rate = y_val.mean()
val_deciles["overall_rate"] = overall_val_rate
test_deciles["overall_rate"] = overall_val_rate

print("Validation deciles (~10% of eligible val customers each; decile 1 = highest scores):")
display(val_deciles)
print("Test deciles (same score floors as validation):")
display(test_deciles)

decile_confidence = val_deciles.set_index("decile")["conversion_rate"].to_dict()
print("Decile -> confidence (validation conversion rate, used on test):")
print(decile_confidence)
print("Decile score floors (validation):", decile_mins)


Validation deciles (edges from validation scores):


,split,decile,n,conversion_rate,lift,score_min,score_max,overall_rate
0,val,1,285,0.035088,2.941176,0.585844,0.763923,0.01193
1,val,2,285,0.024561,2.058824,0.513503,0.585402,0.01193
2,val,3,285,0.017544,1.470588,0.436155,0.513408,0.01193
3,val,4,285,0.017544,1.470588,0.359687,0.436138,0.01193
4,val,5,285,0.007018,0.588235,0.301771,0.359489,0.01193
5,val,6,285,0.010526,0.882353,0.247833,0.301627,0.01193
6,val,7,285,0.003509,0.294118,0.199851,0.247679,0.01193
7,val,8,298,0.003356,0.281287,0.115588,0.199711,0.01193
8,val,9,500,0.000000,0.000000,0.115575,0.115575,0.01193
9,val,10,57,0.000000,0.000000,0.103156,0.115387,0.01193


Test deciles (same score edges as validation):


,split,decile,n,conversion_rate,lift,score_min,score_max,overall_rate
0,test,1,290,0.041379,2.874348,0.585500,0.763852,0.01193
1,test,2,274,0.025547,1.774613,0.513524,0.585307,0.01193
2,test,3,308,0.012987,0.902122,0.436147,0.512634,0.01193
3,test,4,279,0.025090,1.742810,0.359577,0.435039,0.01193
4,test,5,265,0.018868,1.310630,0.302198,0.359295,0.01193
5,test,6,316,0.009494,0.659463,0.247969,0.301641,0.01193
6,test,7,246,0.004065,0.282372,0.200321,0.247494,0.01193
7,test,8,299,0.003344,0.232319,0.115588,0.199471,0.01193
8,test,9,513,0.000000,0.000000,0.115575,0.115583,0.01193
9,test,10,58,0.017241,1.197645,0.099305,0.115387,0.01193


Decile -> confidence (validation conversion rate, used on test):
{1: 0.03508771929824561, 2: 0.02456140350877193, 3: 0.017543859649122806, 4: 0.017543859649122806, 5: 0.007017543859649123, 6: 0.010526315789473684, 7: 0.0035087719298245615, 8: 0.003355704697986577, 9: 0.0, 10: 0.0}


In [16]:
test_scored = test_df[["customer_id"]].copy()
test_scored["propensity_score"] = p_test
test_scored["decile"] = test_decile
test_scored["propensity_confidence"] = test_scored["decile"].map(decile_confidence)
test_scored["actual"] = y_test
test_scored.head(10)


,customer_id,propensity_score,decile,propensity_confidence,actual
6,C119488,0.511566,3,0.017544,0
16,C111819,0.320001,5,0.007018,0
23,C113243,0.553962,2,0.024561,0
28,C102829,0.433485,4,0.017544,1
33,C116655,0.466384,3,0.017544,0
38,C113266,0.256078,6,0.010526,0
40,C111850,0.585613,1,0.035088,0
46,C110539,0.555694,2,0.024561,0
54,C117700,0.158578,8,0.003356,0
60,C110178,0.311836,5,0.007018,0


In [22]:
if USE_XGB:
    model_path = ARTIFACT_PREFIX.with_suffix(".json")
    model.save_model(model_path)
else:
    model_path = ARTIFACT_PREFIX.with_suffix(".joblib")
    joblib.dump(model, model_path)

preprocess_path = ARTIFACT_PREFIX.with_name(ARTIFACT_PREFIX.name + "_preprocess.joblib")
decile_path = ARTIFACT_PREFIX.with_name(ARTIFACT_PREFIX.name + "_decile_mins.npy")
confidence_path = ARTIFACT_PREFIX.with_name(ARTIFACT_PREFIX.name + "_decile_confidence.json")
metrics_path = ARTIFACT_PREFIX.with_name(ARTIFACT_PREFIX.name + "_metrics.csv")

joblib.dump(
    {
        "num_cols": NUM_COLS,
        "cat_cols": CAT_COLS,
        "medians": train_medians,
        "dummy_columns": dummy_columns,
        "label_col": LABEL_COL,
        "elig_col": ELIG_COL,
        "threshold": THRESHOLD,
        "use_xgb": USE_XGB,
    },
    preprocess_path,
)
np.save(decile_path, decile_mins)
pd.Series(decile_confidence).to_json(confidence_path)
metrics_df.to_csv(metrics_path)
test_scored.to_csv(ARTIFACT_PREFIX.with_name(ARTIFACT_PREFIX.name + "_test_scored.csv"), index=False)

print("Saved:")
print(model_path)
print(preprocess_path)
print(decile_path)
print(confidence_path)


Saved:
/Users/nikhilsingh.parihar/Documents/Megathon/propensity_model_outputs/ev_finance.joblib
/Users/nikhilsingh.parihar/Documents/Megathon/propensity_model_outputs/ev_finance_preprocess.joblib
/Users/nikhilsingh.parihar/Documents/Megathon/propensity_model_outputs/ev_finance_decile_edges.npy
/Users/nikhilsingh.parihar/Documents/Megathon/propensity_model_outputs/ev_finance_decile_confidence.json


In [ ]:
SHAP_SAMPLE = min(500, len(X_test))
rng = np.random.default_rng(42)
shap_idx = rng.choice(len(X_test), size=SHAP_SAMPLE, replace=False)
X_shap = X_test.iloc[shap_idx].astype(np.float64)

if USE_XGB:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_shap)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
else:
    bg_n = min(100, len(X_train))
    background = shap.sample(X_train.astype(np.float64), bg_n, random_state=42)
    explainer = shap.Explainer(
        lambda x: model.predict_proba(x)[:, 1],
        background,
    )
    shap_values = explainer(X_shap.values).values

mean_abs_shap = pd.DataFrame(
    {
        "feature": X_shap.columns,
        "mean_abs_shap": np.abs(shap_values).mean(axis=0),
    }
).sort_values("mean_abs_shap", ascending=False)

print("Top 20 features by mean |SHAP|:")
display(mean_abs_shap.head(20))

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, show=False, max_display=20)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fpr, tpr, _ = roc_curve(y_test, p_test)
roc_auc = roc_auc_score(y_test, p_test)
axes[0].plot(fpr, tpr, label=f"test (AUC={roc_auc:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", linewidth=1, label="random")
axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC curve (test)")
axes[0].legend(loc="lower right")
axes[0].grid(True, alpha=0.3)

precision, recall, _ = precision_recall_curve(y_test, p_test)
pr_auc = average_precision_score(y_test, p_test)
axes[1].plot(recall, precision, label=f"test (AP={pr_auc:.3f})")
axes[1].axhline(y_test.mean(), color="k", linestyle=":", linewidth=1, label=f"baseline ({y_test.mean():.3f})")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision–recall curve (test)")
axes[1].legend(loc="upper right")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
